# Deriving the migration trigger

**Phase 0 exit artifact** (`PLAN.md` → Phase 0; design note §3). **Status: skeleton.**

This notebook has one job: show that the pre-emptive migration trigger in
`HazardAwarePolicy.should_migrate` *falls out of* the expected-cost function
$E = T + L$, rather than being a separate heuristic bolted next to it. Each
section below is either a derivation to write (marked **TODO**) or a numerical
check that already runs.

What must be true by the end:

1. $E_i(r) = T_i + L_i$ is derived, not asserted (§2).
2. "migrate iff the risk removed pays for the move" is the same objective, not a new one (§3).
3. The 16-cell Riemann sum used in production is accurate enough to trust (§2, checked below).
4. The hysteresis $h$ has a stated failure mode on each side (§4).
5. Everything §2–§5 could not settle is copied into design note §8.

In [ ]:
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path.cwd().parent / "src"))  # run from notebooks/ without installing

from hazardserve.cost import MigrationCost, NodeSpec, Request
from hazardserve.crossover import crossover_table, crossover_tokens
from hazardserve.hazard import HazardModel, NodeHistory
from hazardserve.policy import HazardAwarePolicy

rng = np.random.default_rng(1)
NOW = 1e5


def node(name, prefill_tps=250.0, decode_tps=12.0, bandwidth_bps=100e6 / 8):
    return NodeSpec(name, decode_tps=decode_tps, prefill_tps=prefill_tps, bandwidth_bps=bandwidth_bps,
                    kv_bytes_per_token=128e3, mem_tokens=200_000)


def model(sessions, age, name="spot"):
    "A hazard model whose node has `sessions` of history and is `age` seconds into a live session."
    m = HazardModel(min_sessions=5)
    m.history[name] = NodeHistory(sessions=list(sessions), current_start=NOW - age)
    return m


REQ = Request(rid=0, prompt_tokens=4000, expected_output_tokens=2500, true_output_tokens=2500, arrival=0.0)
SESSIONS = 1200.0 * rng.weibull(3.0, 800)  # increasing-hazard archetype, ~18 min mean

## 1. Setup and notation

| symbol | meaning | where it comes from |
|---|---|---|
| $a_i$ | node $i$'s uptime so far | `HazardModel.age` |
| $\bar F_i(t \mid a_i)$ | conditional survival | `SurvivalCurve.conditional_S` |
| $T_i$ | queue wait + remaining service time | `cost.service_time` + load |
| $U_i(t)$ | cost of an unplanned loss at $t$ | recompute on the best fallback + $\sigma$ |
| $L_i$ | $\int_0^{T_i} f_i(t\mid a_i)\,U_i(t)\,dt$ | `HazardModel.expected_loss_integral` |
| $h$ | hysteresis | `HazardAwarePolicy.hysteresis` |

**TODO (design note §2):** state the queueing model precisely enough that $T_i$ is well defined,
and say what is assumed known ($P_r$, noisy $\hat L_r$, node rates, bandwidth).

## 2. The expected completion cost, derived

**TODO.** Condition on the failure time and take expectations; show the sketch below is the
first-order term and name what is dropped (the request may fail *again* on the fallback node —
the model truncates the recursion at one loss).

$$
E_i(r) \;=\; T_i \;+\; \int_0^{T_i} f_i(t \mid a_i)\, U_i(t)\, dt
$$

The integral is computed as a 16-cell Riemann sum. The check below says how much that costs in accuracy.

In [ ]:
def monte_carlo_loss(m, spec, age, T, draws=200_000, slo_penalty=10.0, mig=MigrationCost()):
    "Sample the failure time from the conditional survival curve and average the realised loss."
    curve = m.curve("spot", NOW)
    grid = np.linspace(0.0, T, 4001)
    F = 1.0 - np.asarray(curve.conditional_S(grid, age))  # conditional CDF over the horizon
    u = rng.random(draws)
    t_fail = np.interp(u, F, grid)
    inside = u <= F[-1]                                    # draws that fail before the request finishes
    ctx = REQ.prompt_tokens + np.clip(t_fail * spec.decode_tps, 0, REQ.expected_output_tokens)
    loss = mig.recompute_overhead + ctx / spec.prefill_tps + slo_penalty
    return float(np.mean(np.where(inside, loss, 0.0))), float(F[-1])


age = 600.0
m = model(SESSIONS, age)
spec = node("spot")
nodes = {"spot": spec, "fallback": spec}
policy = HazardAwarePolicy(m, slo_penalty=10.0)
T, L = policy.cost_components(REQ, "spot", REQ.prompt_tokens, 0, nodes, dict.fromkeys(nodes, 0.0), NOW)
mc, p_fail = monte_carlo_loss(m, spec, age, T)
print(f"T = {T:8.2f} s")
print(f"L  16-cell Riemann = {L:6.3f} s")
print(f"L  Monte Carlo     = {mc:6.3f} s   (relative error {abs(mc - L) / L:.2%})")
print(f"P(node leaves before the request finishes | uptime {age/60:.0f} min) = {p_fail:.3f}")

In [ ]:
# How many cells does the integral actually need? The policy ships 16.
for steps in (4, 8, 16, 32, 64, 256):
    p = HazardAwarePolicy(m, slo_penalty=10.0, integral_steps=steps)
    _, li = p.cost_components(REQ, "spot", REQ.prompt_tokens, 0, nodes, dict.fromkeys(nodes, 0.0), NOW)
    print(f"steps={steps:4d}   L={li:7.4f}   error vs MC={abs(li - mc) / mc:7.3%}")

## 3. From $E$ to the migration trigger

**TODO — the core of this notebook.** A request already running on $i$ can stay or move to $j$ at
planned cost $C_{ij}$. Staying costs $T_i + L_i$; moving costs $C_{ij} + T_j + L_j$. Minimising the
same $E$ gives *move iff* $E_i > C_{ij} + E_j$, yet the shipped rule is the stricter pair

$$
\underbrace{L_i - (C_{ij} + L_j)}_{\text{risk gain}} > h
\qquad\text{and}\qquad
\underbrace{(T_j + C_{ij}) - T_i}_{\Delta T} < h .
$$

Show that this is the same stationary point restricted to the sub-problem where the *time* terms are
a wash — i.e. that splitting $E$ into $T$ and $L$ and requiring the gain to come from $L$ is what keeps
the migration path from acting as a load balancer on noisy inputs (the thrashing recorded in
`docs/algorithm.md`). State precisely what is given up by the restriction: migrations that would be
profitable on $T$ alone are never taken.

In [ ]:
# Where does the trigger currently fire, and when it does not, which of the two
# conditions blocks it? A spot node deep into its life vs a fresh one.
old, fresh = node("spot"), node("fresh")
two = {"spot": old, "fresh": fresh}
loads = dict.fromkeys(two, 0.0)
done, h = 500, 2.0
for age_min in (2, 10, 20, 30):
    m2 = HazardModel(min_sessions=5)
    m2.history["spot"] = NodeHistory(sessions=list(SESSIONS), current_start=NOW - age_min * 60.0)
    m2.history["fresh"] = NodeHistory(sessions=list(SESSIONS), current_start=NOW - 60.0)
    p = HazardAwarePolicy(m2, slo_penalty=10.0, hysteresis=h)
    ctx = REQ.prompt_tokens + done
    t_stay, l_stay = p.cost_components(REQ, "spot", ctx, done, two, loads, NOW)
    t_dest, l_dest = p.cost_components(REQ, "fresh", ctx, done, two, loads, NOW)
    c_mig, method = p.mig.planned(ctx, old, fresh)
    risk_gain = l_stay - (c_mig + l_dest)
    delta_t = (t_dest + c_mig) - t_stay
    decision = p.should_migrate(REQ, "spot", ctx, done, two, loads, NOW)
    print(f"uptime {age_min:2d} min  L_stay={l_stay:6.2f}  L_dest={l_dest:5.2f}  C={c_mig:6.2f} ({method})"
          f"  risk_gain={risk_gain:7.2f} (>{h})  dT={delta_t:6.2f} (<{h})  ->  {decision or 'stay'}")

The two conditions are not symmetric, and the printout above is the reason the v0 trigger barely
fires: a planned move costs a full re-prefill of the context ($C \approx c/p_{\text{dst}}$ over a
WAN-class link), which both eats the risk gain *and* shows up again in $\Delta T$. The trigger can
only fire once $L_i$ has grown past the cost of moving — late in a node's life, on a long request,
or when the link is fast enough for KV transfer to beat re-prefill.

**TODO:** decide whether that is the honest answer (pre-emptive migration pays only in a
characterisable regime — then characterise it) or a mis-specification of $C$: the destination can
prefill *while* the source keeps decoding, so charging the full re-prefill to $\Delta T$ may be
wrong. This is the most consequential open question in Phase 0.

**TODO:** turn the cell above into the Pareto curve `PLAN.md` Phase 1 asks for (planned vs unplanned
migrations as $h$, `check_interval` and `min_remaining` vary), once the simulator sweep exists.

## 4. Hysteresis: what breaks on each side

**TODO (design note §3).**

- $h$ too low → thrashing; the migration path degenerates into a load balancer fed by a noisy $\hat L_r$.
- $h$ too high → the trigger never fires and the policy is placement-only, which is the v0 behaviour.
- Needed: a defensible default, and the sensitivity of the headline result to it.

## 5. Transfer vs recompute

The planned-migration cost $C_{ij}$ is $\min(\text{transfer}, \text{recompute})$; both are affine in the
context length, so they cross at most once (`hazardserve.crossover`). Constants below are *assumed*,
not measured — Phase 4 replaces them.

In [ ]:
# The crossover moves with the destination's prefill rate as much as with bandwidth:
# a slow prefill engine makes shipping KV attractive at short contexts.
for label, prefill_tps in (("datacentre GPU", 3000.0), ("edge / consumer GPU", 250.0)):
    table = crossover_table(node("dst", prefill_tps=prefill_tps), kv_bytes_per_token=128e3)
    cells = [f"{link}: " + ("never" if np.isinf(tok) else f"{tok:,.0f} tok") for link, tok in table.items()]
    print(f"{label:20s} prefill={prefill_tps:6.0f} tok/s   " + " | ".join(cells))

# Sanity: the crossover agrees with the cost model the policy actually calls.
lan = node("lan", bandwidth_bps=10e9 / 8, prefill_tps=3000.0)
c = crossover_tokens(lan, lan)
mig = MigrationCost()
print(f"\njust below c*={c:,.0f}: {mig.planned(int(c * 0.9), lan, lan)[1]}")
print(f"just above c*={c:,.0f}: {mig.planned(int(c * 1.1), lan, lan)[1]}")

## 6. Open questions → design note §8

**TODO.** Copy anything unresolved above into `paper/design-note.md` §8 before ticking the Phase 0 box in `PLAN.md`:

- [ ] Is one-loss truncation in §2 defensible, or does the recursion matter at high churn?
- [ ] Does Kaplan-Meier's independent-censoring assumption survive the scheduler steering work toward long-lived nodes?
- [ ] Which of $h$, `check_interval`, `min_remaining` actually moves the headline number?